# Análise de Componentes Principais (PCA)

Fotografar uma cadeira é projetar um objeto 3D num plano 2D, e alguns ângulos mostram muito mais da cadeira do que outros: de frente aparecem encosto, assento e pernas; de cima, só um quadrado. O PCA faz o mesmo com dados. Ele procura as direções em que os dados aparecem **mais espalhados**, porque é nelas que está a maior parte da informação, e descarta as demais, reduzindo o número de dimensões com pouca perda.

Neste notebook construímos essa ideia testando direções à mão num exemplo 2D, vemos como a álgebra linear encontra a melhor direção de uma vez, implementamos a classe do zero e a aplicamos ao Iris, com 4 dimensões, e a fotos de rostos, com 1850.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_lfw_people, load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)

## A Melhor Sombra

Começamos com 200 pontos em 2D espalhados em torno de uma reta. As duas coordenadas andam juntas: sabendo a primeira, quase sabemos a segunda. Os dados são "quase 1D", e é essa redundância que o PCA vai explorar.

In [ ]:
x1 = rng.normal(0, 3, 200)
x2 = 0.8 * x1 + rng.normal(0, 1, 200)
X = np.c_[x1, x2]

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.6)
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.title('Dados 2D correlacionados')
plt.axis('equal')
plt.show()

Suponha que só possamos guardar **um número** por ponto. Uma forma de fazer isso é escolher uma reta passando pelo centro dos dados e "derrubar" cada ponto sobre ela, guardando apenas a posição onde ele caiu. Isso é uma **projeção**. Para decidir qual reta usar, testamos quatro direções e medimos a **variância** das posições projetadas (em vermelho), que diz o quanto elas ficaram espalhadas.

In [ ]:
X_centered = X - X.mean(axis=0)


def project(X_centered, angle):
    # Projeta os pontos na reta que passa pela origem com o ângulo dado
    direction = np.array([np.cos(angle), np.sin(angle)])
    t = X_centered @ direction            # posição de cada ponto ao longo da reta
    return t, np.outer(t, direction)      # coordenada 1D e ponto projetado no plano


fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, degrees in zip(axes, [0, 40, 90, 135]):
    t, X_proj = project(X_centered, np.radians(degrees))

    for p, q in zip(X_centered, X_proj):
        ax.plot([p[0], q[0]], [p[1], q[1]], color='gray', lw=0.5, alpha=0.5)
    ax.scatter(X_centered[:, 0], X_centered[:, 1], s=10, alpha=0.4)
    ax.scatter(X_proj[:, 0], X_proj[:, 1], s=10, color='crimson')

    ax.set_title(f'{degrees}°   variância = {t.var():.1f}')
    ax.set_xlim(-11, 11)
    ax.set_ylim(-11, 11)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

Em 40° os pontos projetados continuam bem espalhados e distinguíveis entre si; em 135° eles se amontoam e quase toda a informação se perde. Os traços cinza mostram o quanto cada ponto se moveu para cair na reta, e eles são menores justamente onde a variância é maior. Pelo teorema de Pitágoras, **maximizar a variância da projeção e minimizar o erro da projeção são o mesmo critério**.

Testando todos os ângulos de 0° a 180°, a melhor direção fica perto de 40°. A direção de máxima variância é o **primeiro componente principal (PC1)**. O segundo componente, PC2, é a direção de maior variância entre as perpendiculares ao PC1, o que em 2D deixa uma única opção; em mais dimensões o processo continua, com o PC3 perpendicular aos dois primeiros e assim por diante. Testar ângulos funciona em 2D, mas não em 100 dimensões, e por isso precisamos de um atalho.

## Fundamentação Matemática

### Covariância

A **covariância** entre duas variáveis mede se elas crescem juntas (positiva), em sentidos opostos (negativa) ou sem relação linear (perto de zero). A **matriz de covariância** $\Sigma$ reúne a covariância de todos os pares de variáveis, com as variâncias de cada uma na diagonal. Para dados centralizados, com média zero em cada coluna, e $m$ amostras:

$$ \Sigma = \frac{1}{m-1} X^\top X $$

Centralizar é obrigatório, porque queremos descrever o **formato** da nuvem de pontos e não onde ela está.

In [ ]:
cov = np.cov(X_centered, rowvar=False)   # rowvar=False: cada coluna é uma variável
cov

### Autovetores e Autovalores

Um **autovetor** $v$ de uma matriz é uma direção que a matriz não gira, só estica, por um fator $\lambda$ chamado **autovalor**:

$$ \Sigma v = \lambda v $$

O resultado que sustenta o PCA é que, para a matriz de covariância, os autovetores são exatamente as direções dos componentes principais, perpendiculares entre si, e cada autovalor é a variância dos dados ao longo do seu autovetor. Basta então calcular os autovetores e ordená-los do maior autovalor para o menor. Usamos `np.linalg.eigh`, a versão do NumPy para matrizes simétricas.

In [ ]:
eigenvalues, eigenvectors = np.linalg.eigh(cov)

order = np.argsort(eigenvalues)[::-1]     # eigh devolve em ordem crescente
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]     # cada coluna é um autovetor

pc1_angle = np.degrees(np.arctan2(eigenvectors[1, 0], eigenvectors[0, 0])) % 180
print(f"Autovalores: {eigenvalues}")
print(f"Ângulo do PC1: {pc1_angle:.1f}°")

O ângulo do PC1, perto de 40°, e o maior autovalor batem com a melhor direção da figura anterior; a pequena diferença na variância vem do fator $m-1$ contra $m$. Desenhando os dois componentes sobre os dados, com comprimento proporcional ao desvio padrão em cada direção (o sentido da seta não importa, só a direção):

In [ ]:
mean = X.mean(axis=0)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], alpha=0.4)

for i, color in enumerate(['crimson', 'darkorange']):
    arrow = eigenvectors[:, i] * 2 * np.sqrt(eigenvalues[i])
    plt.annotate('', xy=mean + arrow, xytext=mean - arrow,
                 arrowprops=dict(arrowstyle='<->', color=color, lw=3))
    plt.text(*(mean + arrow * 1.15), f'PC{i + 1}', color=color, fontsize=14, weight='bold')

plt.title('Os componentes principais')
plt.axis('equal')
plt.show()

### Variância Explicada

Como cada autovalor é a variância ao longo de um componente, a fração da variância total que cada componente carrega é imediata:

$$ \text{variância explicada}_k = \frac{\lambda_k}{\sum_j \lambda_j} $$

Esse é o número usado para decidir quantos componentes manter.

In [ ]:
eigenvalues / eigenvalues.sum()

O PC1 sozinho carrega quase toda a variância, e o PC2 é praticamente o ruído que adicionamos ao gerar os dados.

## Implementação

A classe segue a receita da seção anterior: centralizar, calcular a covariância, obter os autovetores ordenados, manter os $k$ primeiros e projetar. Além de `fit` e `transform`, incluímos `inverse_transform`, que faz o caminho de volta e reconstrói um ponto no espaço original a partir das coordenadas reduzidas.

In [ ]:
class PCA:
    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self, X):
        self.mean = X.mean(axis=0)
        X_centered = X - self.mean

        cov = np.cov(X_centered, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
        order = np.argsort(eigenvalues)[::-1]
        eigenvalues, eigenvectors = eigenvalues[order], eigenvectors[:, order]

        # um componente por linha
        self.components = eigenvectors[:, :self.n_components].T
        self.explained_variance_ratio = eigenvalues[:self.n_components] / eigenvalues.sum()
        return self

    def transform(self, X):
        # coordenadas de cada ponto nos novos eixos
        return (X - self.mean) @ self.components.T

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def inverse_transform(self, Z):
        # volta para o espaço original
        return Z @ self.components + self.mean

A figura abaixo resume o que o PCA faz com os dados. Com 2 componentes nada se perde: os dados são apenas **girados** para que o PC1 vire o eixo horizontal, e as duas coordenadas deixam de ser correlacionadas. Com 1 componente, guardamos um único número por ponto, e a reconstrução com `inverse_transform` coloca todos os pontos sobre a reta do PC1; a distância até o original é a informação perdida.

In [ ]:
Z_2d = PCA(n_components=2).fit_transform(X)

pca_1d = PCA(n_components=1).fit(X)
X_rec = pca_1d.inverse_transform(pca_1d.transform(X))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].scatter(X[:, 0], X[:, 1], alpha=0.5)
axes[0].set(title='Original', xlabel='$x_1$', ylabel='$x_2$')

axes[1].scatter(Z_2d[:, 0], Z_2d[:, 1], alpha=0.5, color='seagreen')
axes[1].set(title='Nos novos eixos (2 componentes)', xlabel='PC1', ylabel='PC2')

for p, q in zip(X, X_rec):
    axes[2].plot([p[0], q[0]], [p[1], q[1]], color='gray', lw=0.5)
axes[2].scatter(X[:, 0], X[:, 1], alpha=0.2)
axes[2].scatter(X_rec[:, 0], X_rec[:, 1], s=10, color='crimson')
axes[2].set(title='Só o PC1, reconstruído', xlabel='$x_1$', ylabel='$x_2$')

for ax in axes:
    ax.set_xlim(-11, 11)
    ax.set_ylim(-8, 8)
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

### Comparação com o Scikit-Learn

```python
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(X)
pca.transform(X)                  # projeção
pca.components_                   # componentes, um por linha
pca.explained_variance_ratio_     # fração da variância por componente
pca.inverse_transform(Z)          # reconstrução
```

A interface é a mesma, com os atributos terminados em `_`. Internamente a biblioteca não calcula a matriz de covariância: aplica a **decomposição em valores singulares (SVD)** diretamente sobre os dados centralizados, o que dá o mesmo resultado com mais estabilidade numérica.

## Aplicação: Iris

O Iris tem 4 medidas por flor, o que impede de ver os dados num gráfico. Antes de projetá-lo em 2D, **padronizamos** as colunas para média 0 e desvio 1: o PCA procura variância, e uma medida expressa em números grandes dominaria os componentes só por causa da unidade.

In [ ]:
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
y_iris = iris.target

ratios = PCA(n_components=4).fit(X_iris).explained_variance_ratio

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 5), ratios, color='steelblue', label='por componente')
ax.plot(range(1, 5), np.cumsum(ratios), 'o-', color='crimson', label='acumulada')
ax.set(xlabel='componente', ylabel='variância explicada', xticks=range(1, 5), ylim=(0, 1.05))
ax.legend()
plt.show()

Esse gráfico de barras decrescentes é o **scree plot**, a ferramenta usual para escolher o número de componentes. Os dois primeiros guardam quase 96% da variância, então reduzir de 4 para 2 dimensões custa pouco. Ao lado da projeção, mostramos os **loadings**, o peso de cada medida original em cada componente, que explicam o que os novos eixos significam.

In [ ]:
pca_iris = PCA(n_components=2).fit(X_iris)
Z_iris = pca_iris.transform(X_iris)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={'width_ratios': [1.3, 1]})

for i, name in enumerate(iris.target_names):
    axes[0].scatter(Z_iris[y_iris == i, 0], Z_iris[y_iris == i, 1], label=name, alpha=0.8)
axes[0].set(title='Iris projetado em 2D', xlabel='PC1', ylabel='PC2')
axes[0].legend()

positions = np.arange(4)
axes[1].barh(positions - 0.2, pca_iris.components[0], height=0.4, label='PC1')
axes[1].barh(positions + 0.2, pca_iris.components[1], height=0.4, label='PC2')
axes[1].set(yticks=positions, yticklabels=iris.feature_names, title='Peso de cada medida')
axes[1].axvline(0, color='black', lw=0.8)
axes[1].legend()

plt.tight_layout()
plt.show()

O PCA **não usou os rótulos**: ele só procurou variância, e as espécies apareceram separadas ao longo do PC1 como consequência. Isso não é garantido em geral, porque a direção de maior variância pode vir de algo irrelevante para a estrutura que interessa.

Pelos loadings, o PC1 é um eixo de **tamanho da flor**, que combina as medidas que crescem juntas, e o PC2 é quase só a largura da sépala. O sinal de cada componente é arbitrário, então outra implementação pode devolver os eixos espelhados.

## Aplicação: Eigenfaces

Uma imagem em tons de cinza é um vetor com um número por pixel. As fotos do *Labeled Faces in the Wild* que usamos têm 50×37 = 1850 pixels, então cada rosto é um ponto em 1850 dimensões. Como rostos alinhados são muito parecidos entre si, eles ocupam uma região pequena desse espaço, e é essa região que o PCA encontra. O download acontece só na primeira execução.

In [ ]:
faces = fetch_lfw_people(min_faces_per_person=70, resize=0.4)
X_faces = faces.data
h, w = faces.images.shape[1:]

print(f"{X_faces.shape[0]} fotos de {h}x{w} = {X_faces.shape[1]} pixels cada")

In [ ]:
def show_images(images, titles, n_cols=6):
    n_rows = int(np.ceil(len(images) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.8 * n_cols, 2.3 * n_rows))
    for ax, img, title in zip(np.ravel(axes), images, titles):
        ax.imshow(img.reshape(h, w), cmap='gray')
        ax.set_title(title, fontsize=10)
    for ax in np.ravel(axes):
        ax.axis('off')
    plt.tight_layout()
    plt.show()


show_images(X_faces[:6], [faces.target_names[t].split()[-1] for t in faces.target[:6]])

Ajustamos o PCA **apenas no treino**, guardando 150 componentes, para depois reconstruir rostos de teste que ele nunca viu. Cada componente é um vetor de 1850 números e pode ser desenhado como imagem: são as **eigenfaces**.

In [ ]:
X_train, X_test = train_test_split(X_faces, test_size=0.25, random_state=42)

pca_faces = PCA(n_components=150).fit(X_train)
print(f"Variância retida pelos 150 componentes: {pca_faces.explained_variance_ratio.sum():.1%}")

show_images([pca_faces.mean] + list(pca_faces.components[:11]),
            ['rosto médio'] + [f'eigenface {i + 1}' for i in range(11)])

As eigenfaces não são pessoas, e sim **padrões de variação**, como "mais luz à esquerda" ou "rosto mais largo". Qualquer rosto pode ser escrito como o rosto médio mais uma combinação delas,

$$ \text{rosto} \approx \text{rosto médio} + z_1 \cdot \text{eigenface}_1 + z_2 \cdot \text{eigenface}_2 + \dots + z_k \cdot \text{eigenface}_k $$

em que os pesos $z_1, \dots, z_k$ são as coordenadas calculadas pelo `transform`. As primeiras eigenfaces capturam principalmente **iluminação**: em fotos tiradas sem controle, essa é a maior fonte de variação, mesmo sem relação com a identidade da pessoa. Reconstruindo um rosto de teste com cada vez mais componentes:

In [ ]:
face = X_test[[3]]
ks = [1, 5, 20, 50, 100, 150]

reconstructions = []
for k in ks:
    pca_k = PCA(n_components=k).fit(X_train)
    reconstructions.append(pca_k.inverse_transform(pca_k.transform(face))[0])

show_images(reconstructions + [face[0]], [f'k = {k}' for k in ks] + ['original'], n_cols=7)

Com 1 componente aparece apenas um rosto genérico com a iluminação certa; com 20 a pessoa começa a ser reconhecível; com 150, guardando **12 vezes menos números** que a imagem original, o rosto está claro e só um pouco suavizado. O PCA preserva o que é comum a muitos rostos e descarta detalhes finos, como textura de pele e fios de cabelo.

## Custo e Limitações

Montar a matriz de covariância custa $O(m n^2)$ e diagonalizá-la, $O(n^3)$, com $n$ features e $m$ amostras. Para os 1850 pixels do LFW isso é rápido; com centenas de milhares de features, a matriz nem cabe na memória e é preciso recorrer à SVD truncada ou ao `IncrementalPCA`.

A limitação de fundo é que o PCA só enxerga **estruturas lineares**, porque procura um plano para projetar os dados. Se eles vivem sobre uma superfície curva, como uma folha enrolada em espiral, nenhuma projeção plana consegue desenrolá-la; isso exige métodos **não lineares**, assunto dos próximos notebooks.

Além disso, **variância não é sinônimo de importância**: nas eigenfaces, os primeiros componentes descrevem iluminação, não identidade, e uma estrutura de baixa variância é a primeira a ser descartada. O método também é sensível à escala, já que sem padronização a variável com números maiores domina, e a outliers, porque a variância usa desvios ao quadrado e poucos pontos extremos podem girar os componentes. Por fim, cada componente mistura todas as variáveis originais, o que torna a interpretação difícil quando elas são muitas.

## Exercícios

### Exercício 1: Vinhos

O dataset `load_wine` tem 13 medidas químicas de vinhos de 3 cultivares. Padronize os dados, faça o scree plot com os 13 componentes e projete em 2D colorindo pelo cultivar. Quanto da variância os 2 primeiros componentes guardam, comparado aos 96% do Iris? Quais medidas químicas mais pesam no PC1?

In [ ]:
# Seu código aqui

### Exercício 2: Dígitos

O dataset `load_digits` tem imagens de dígitos manuscritos de 8×8 pixels. Repita a análise das eigenfaces: desenhe os primeiros componentes como imagens e reconstrua alguns dígitos com 2, 5, 10 e 30 componentes. Quantos componentes são necessários para guardar 90% da variância? Projete em 2D colorindo pelo dígito e identifique quais dígitos se misturam.

In [ ]:
# Seu código aqui

### Exercício 3: Dados sobre um Plano

Gere 300 pontos em 3D próximos de um plano, por exemplo $x_3 = x_1 + 2x_2 + \text{ruído}$, e aplique a classe `PCA` com 3 componentes. O que o terceiro autovalor diz sobre os dados? Compare o terceiro componente com o vetor normal ao plano, $(1, 2, -1)$, normalizado.

In [ ]:
# Seu código aqui